# Used Car Price Predictor – India

**Project description:** An end-to-end machine learning project that estimates the
listing-market value of used cars in India and checks whether an asking price is fair.

**Problem statement:** Used-car prices depend on age, usage, fuel, transmission, power
and ownership, but buyers and sellers have no quick, evidence-based way to judge a listing.

**Objectives**
1. Clean and analyse real CarDekho listing data.
2. Build and compare regression models, then tune the best one.
3. Give a price estimate with a realistic range and a SHAP explanation.
4. Compare an asking price with the estimate (within, above or below range).
5. Deploy everything as a Streamlit app.

**Business use case:** Buyers avoid overpaying, sellers price competitively, and
platforms or dealers get a data-driven price reference.

**Why this problem:** Real, messy data; a clear target; explainable results; easy to demo.

**Important:** The output is an *estimated listing-market value*, not a verified sale price.

In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("used-car-price-predictor/data/raw/Car details v3.csv")
print("Shape:", df.shape)
df.head()

Shape: (8128, 13)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB


In [17]:
df.describe()

,year,selling_price,km_driven,seats
count,8128.000000,8.128000e+03,8.128000e+03,7907.000000
mean,2013.804011,6.382718e+05,6.981951e+04,5.416719
std,4.044249,8.062534e+05,5.655055e+04,0.959588
min,1983.000000,2.999900e+04,1.000000e+00,2.000000
25%,2011.000000,2.549990e+05,3.500000e+04,5.000000
50%,2015.000000,4.500000e+05,6.000000e+04,5.000000
75%,2017.000000,6.750000e+05,9.800000e+04,5.000000
max,2020.000000,1.000000e+07,2.360457e+06,14.000000


In [18]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
 name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        215
torque           222
seats            221
dtype: int64

Duplicate rows: 1202


In [19]:
for col in ["fuel", "seller_type", "transmission", "owner"]:
    print(f"\n{col}:\n{df[col].value_counts()}")


fuel:
fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

seller_type:
seller_type
Individual          6766
Dealer              1126
Trustmark Dealer     236
Name: count, dtype: int64

transmission:
transmission
Manual       7078
Automatic    1050
Name: count, dtype: int64

owner:
owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64


In [20]:
# Text columns that will need cleaning
df[["name", "mileage", "engine", "max_power", "torque"]].sample(10, random_state=42)

,name,mileage,engine,max_power,torque
1971,Honda City 1.5 GXI,12.8 kmpl,1493 CC,100 bhp,13.1kgm@ 4600rpm
4664,Tata Safari Storme EX,14.0 kmpl,2179 CC,138.1 bhp,320Nm@ 1700-2700rpm
5448,Maruti Ritz VDi,23.2 kmpl,1248 CC,73.94 bhp,190Nm@ 2000rpm
3333,Honda City 2017-2020 VTEC,16.9 kmpl,1497 CC,100 bhp,"13.5@ 4,800(kgm@ rpm)"
2316,Maruti Swift VDI,22.9 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm
6870,Maruti Alto 800 LX,22.74 kmpl,796 CC,47.3 bhp,69Nm@ 3500rpm
5953,Mahindra Scorpio 1.99 S10 4WD,15.4 kmpl,1997 CC,120 bhp,280Nm@ 1800-2800rpm
2418,Ford Figo Aspire 1.2 Ti-VCT Sports Edition,18.12 kmpl,1196 CC,86.8 bhp,112Nm@ 4000rpm
1811,Maruti Alto LX,19.7 kmpl,796 CC,46.3 bhp,62Nm@ 3000rpm
3608,Maruti Wagon R LXI,18.9 kmpl,998 CC,67.1 bhp,90Nm@ 3500rpm
